# Gelbooru to Hugging Face Dataset Maker 🍋

Scrapes images + tags from [Gelbooru](https://gelbooru.com/), lets you curate/clean the tags, zips the result, and (optionally) uploads the finished dataset straight to a Hugging Face dataset repo.
Note that this is rate limited by default. Please don't abuse Gelbooru. 

Part of [citron_gelbooru_scraper](https://github.com/citronlegacy/citron_gelbooru_scraper) — see the repo for more scraping/tagging tools.

---
### Project Disclaimer
Please read and follow the [Google Colab guidelines](https://research.google.com/colaboratory/faq.html) and its [Terms of Service](https://research.google.com/colaboratory/tos_v3.html).

---
### Setup: Save Your Secrets in Colab

This notebook reads credentials from **Colab Secrets** instead of typing them into form fields. To add a secret:

1. Click the 🔑 **key icon** in the left sidebar of Colab to open the Secrets panel.
2. Click **+ Add new secret**.
3. Enter the secret **Name** exactly as listed below, and paste your value into **Value**.
4. Toggle **Notebook access** ON for each secret so this notebook is allowed to read it.

Add the following secrets (this notebook does not cover how to obtain these values, only how to store them):

| Secret Name | Used For |
|---|---|
| `GELBOORU_API_KEY` | Gelbooru API authentication |
| `GELBOORU_USER_ID` | Gelbooru API authentication |
| `HF_TOKEN` | Uploading the finished dataset to Hugging Face Hub |

---


In [ ]:
#@title # Install
#@markdown Installs `aria2` (used to download images) and defines a small helper
#@markdown used later to count images in the dataset folder.

import os
import time
from pathlib import Path

!apt -y install aria2 -qq

image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp']

root_dir = "/content"

def count_images_in_folder(folder_path):
    # Ensure the folder path is a valid directory
    folder_path = Path(folder_path)
    if not folder_path.is_dir():
        raise ValueError(f"The provided path '{folder_path}' does not exist. If it does exist but Colab can't find it try reconnecting to Google Drive")

    # Count the number of image files in the folder
    image_count = sum(1 for file in folder_path.iterdir() if file.suffix.lower() in image_extensions)

    return image_count


In [ ]:
#@title # 🟣 JellyTag 3.0.6 - Dataset Maker with Gelbooru Tagging

import os
import time
from IPython import get_ipython
from IPython.display import display, Markdown
import json
from urllib.request import urlopen, Request
from collections import Counter
from datetime import datetime
from PIL import Image
import tempfile
import zipfile
from tqdm.auto import tqdm

start_time = time.perf_counter()

image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp']

COLAB = True

if COLAB:
  from google.colab.output import clear as clear_output
else:
  from IPython.display import clear_output

#defining variables at the beginning to fix a bug with the log file - lol such programming
remove_tags = "NA - Tagging Skipped"
topTags = "NA - Tagging Skipped"
total_steps = 0 #Defining this variable here so that its a global varaible

#@markdown ### 1️⃣ Name Project

#@markdown Your project name can't contain spaces
project_name = "project_name" #@param {type:"string"}
project_name = project_name.strip()
# Remove invalid characters from project_name
project_name = "".join(c for c in project_name if c.isalnum() or c in ('_', '-'))

root_dir = "/content"
main_dir = os.path.join(root_dir, "lora_training")

config_folder = os.path.join(main_dir, "config", project_name)
images_folder = os.path.join(main_dir, "datasets", project_name)
zipped_datasets_folder = os.path.join(root_dir, "zipped_datasets") # Defined zipped datasets folder

for dir in [main_dir, images_folder, config_folder]:
  os.makedirs(dir, exist_ok=True)

# Define project_base and project_subfolder here
project_base = project_name if "/" not in project_name else project_name[:project_name.rfind("/")]
project_subfolder = project_name if "/" not in project_name else project_name[project_name.rfind("/")+1:]

#@markdown ---
#@markdown ### 2️⃣ Upload Dataset to Hugging Face Hub
upload_to_huggingface = False #@param {type:"boolean"}
#@markdown Repo id to upload to, e.g. `your-username/your-dataset-name`. The zip is named after the repo (e.g. `pikachu.zip` for `claude/pikachu`).
huggingface_repo_id = "" #@param {type:"string"}
huggingface_repo_private = True #@param {type:"boolean"}
#@markdown Try converting images to lossless WebP before zipping, and keep whichever is smaller. Leave off to always zip the original formats.
convert_to_webp = False #@param {type:"boolean"}


#######################################################
##### STEP - Image Downloading
#######################################################

#@markdown ---
#@markdown ### 3️⃣ Scrape images from [Gelbooru](https://gelbooru.com/)

#@markdown Downloads retry automatically (up to 10 attempts per image, 5s apart) to ride out transient failures or rate-limit blips from Gelbooru's CDN. Any image that still fails after retries is skipped, and its tag file is cleaned up so it doesn't end up orphaned. <p>

#@markdown Gelbooru tags must use underscores not whitespaces. Tags in the curation step can use whitespaces or underscores

tags = "tags" #@param {type:"string"}
extra_tags = "-animated sort:score " #@param {type:"string"}
exclude_these_gelbooru_tags = "game_freak, tagme, creatures_(company),  commentary_request, commentary, english_commentary, symbol-only_commentary, translation_request, translated,  bad pixiv id,  bad id, skeb commission, commission," #@param {type:"string"}

tags = tags + " " +  extra_tags
gelbooruSearchQuery = tags
gelbooruSearchQuery = gelbooruSearchQuery.replace("(", r"\(").replace(")", r"\)")

max_resolution = 3072
include_posts_with_parent = True

tags = tags.replace(" ", "+")\
          .replace("(", "%28")\
          .replace(")", "%29")\
          .replace(":", "%3a")\
          .replace("&", "%26")\

authenticate_with_api_key_to_bypass_401_error = True #@param {type:"boolean"}
#API Doucmentation https://gelbooru.com/index.php?page=wiki&s=view&id=18780
#@markdown Reads credentials from Colab Secrets (key icon in the left sidebar): add secrets named `GELBOORU_API_KEY` and `GELBOORU_USER_ID`, and make sure notebook access is enabled for each.
api_key = ""
user_id = ""
if authenticate_with_api_key_to_bypass_401_error:
  from google.colab import userdata
  try:
    api_key = userdata.get('GELBOORU_API_KEY')
    user_id = userdata.get('GELBOORU_USER_ID')
  except userdata.SecretNotFoundError as e:
    raise RuntimeError(
      "Missing Colab secret. Add GELBOORU_API_KEY and GELBOORU_USER_ID via the "
      "key icon in the left sidebar, or set authenticate_with_api_key_to_bypass_401_error to False."
    ) from e
  except userdata.NotebookAccessError as e:
    raise RuntimeError(
      "GELBOORU_API_KEY / GELBOORU_USER_ID secrets exist but notebook access is disabled. "
      "Enable notebook access for both secrets in the left sidebar."
    ) from e


user_agent = "Mozilla/5.0 AppleWebKit/537.36 (KHTML, like Gecko; compatible; Googlebot/2.1; +http://www.google.com/bot.html) Chrome/93.0.4577.83 Safari/537.36"
limit = 100 # hardcoded by gelbooru
#@markdown Enter maximum number of images to download from Gelbooru (There is a bug where it sometimes downloads 1 more/less than the number entered)
maxNumberOfImages = 100 #@param {type:"number"}
total_limit = maxNumberOfImages

supported_types = (".png", ".jpg", ".jpeg")

def get_json(url):
  print("get_json url = " + url)
  with urlopen(Request(url, headers={"User-Agent": user_agent})) as page:
    return json.load(page)

def clean_tags(directory, blacklisted_tags):
  top_tags = Counter()

  # Iterate over each text file in the specified directory
  for txt_file in [f for f in os.listdir(directory) if f.lower().endswith(".txt")]:
      txt_path = os.path.join(directory, txt_file)

      # Read tags from the text file
      with open(txt_path, 'r') as f:
          tags = [t for t in f.read().split(",")]

          # Replace underscores with whitespace
          tags = [t.replace("_", " ") if len(t) > 3 else t for t in tags]
          # Remove blacklisted tags
          tags = [t for t in tags if t not in blacklisted_tags]

      # Update the Counter with cleaned tags
      top_tags.update(tags)

      # Write cleaned tags back to the text file
      with open(txt_path, 'w') as f:
          f.write(", ".join(tags))

  return top_tags

def filter_images(data):
  image_urls_to_download = []

  for post in data.get("post", []):
      md5 = post.get("md5", "")
      original_image_url = post.get("file_url", "")
      sample_image_url = post.get("sample_url", "")
      post_id = post.get("id", "")

      # Save tags to a file
      import html

      input_string = post.get("tags", "")
      decoded_string = html.unescape(input_string)
      tags = decoded_string.replace("'", "").split()

      #remove tags that are in exclude_these_gelbooru_tags
      tags = [t for t in tags if t not in exclude_these_gelbooru_tags]

      tags_file_path = os.path.join(images_folder, f"{md5}.txt")
      with open(tags_file_path, "w") as tags_file:
          tags_file.write(",".join(tags))

      image_url_to_download = original_image_url

      image_urls_to_download.append(image_url_to_download)

      # Print Gelbooru post link
      gelbooru_post_link = f"https://gelbooru.com/index.php?page=post&s=view&id={post_id}"

  return image_urls_to_download

def download_images():
  count = 0
  if(maxNumberOfImages < 100):
    url = f"https://gelbooru.com/index.php?page=dapi&&api_key={api_key}&user_id={user_id}&json=1&s=post&q=index&limit={maxNumberOfImages+1}&tags={tags}" if authenticate_with_api_key_to_bypass_401_error else f"https://gelbooru.com/index.php?page=dapi&json=1&s=post&q=index&limit={maxNumberOfImages+1}&tags={tags}"
    data = get_json(url)
    count = data["@attributes"]["count"]
  else:
    url = f"https://gelbooru.com/index.php?page=dapi&&api_key={api_key}&user_id={user_id}&json=1&s=post&q=index&limit=100&tags={tags}" if authenticate_with_api_key_to_bypass_401_error else "https://gelbooru.com/index.php?page=dapi&json=1&s=post&q=index&limit=100&tags={}".format(tags)
    data = get_json(url)
    count = data["@attributes"]["count"]


  if count == 0:
    print("📷 No results found")
    return

  print(f"🎯 Found {count} results")
  test_url = "https://gelbooru.com/index.php?page=post&s=list&tags={}".format(tags)
  display(Markdown(f"[Click here to open in browser!]({test_url})"))
  print (f"🔽 Will download to {images_folder} (A confirmation box should appear below, otherwise run this cell again)")

  print("📩 Grabbing image list...")

  image_urls = set()
  image_urls = image_urls.union(filter_images(data))
  for i in range(total_limit // limit):
    print("inside loop")
    numberOfImagesDownloadLinksInFile = len(image_urls)
    numberOfDownloadsRemaining = maxNumberOfImages - numberOfImagesDownloadLinksInFile
    #Debugging log - can be deleted
    print (f"i = {i}; image_urls = {len(image_urls)}; total_limit = {total_limit}; limit = {limit}; numberOfImagesDownloadLinksInFile = {numberOfImagesDownloadLinksInFile}; numberOfDownloadsRemaining = {numberOfDownloadsRemaining} " )
    count -= limit
    if count <= 0:
      break
    time.sleep(0.1)

    # Reformat URLs to ensure only the correct amount of images is downloaded
    #Added if-block to ensure that the last set of images to download doesnt go over the maxNumberOfImages
    #Determine how many download links are remaining
    #If there are less than the limit (hardcoded to 100) then only add the right amount of urls to the downloads list
    numberOfDownloadsRemaining = maxNumberOfImages - numberOfImagesDownloadLinksInFile
    if (numberOfDownloadsRemaining < 100):
      filterImagesResult = filter_images(get_json(url+f"&pid={i+1}"))
      limitedFilterImagesResult = filterImagesResult[0:numberOfDownloadsRemaining]
      image_urls = image_urls.union(limitedFilterImagesResult)
    else:
      filterImagesResult = filter_images(get_json(url+f"&pid={i+1}"))
      image_urls = image_urls.union(filterImagesResult)


  # Skip images that are already sitting in images_folder (e.g. from a
  # previous run) so we don't waste requests re-downloading them.
  already_downloaded_basenames = {os.path.splitext(f)[0] for f in os.listdir(images_folder) if f.lower().endswith(supported_types)}
  urls_to_download = [u for u in image_urls if os.path.splitext(os.path.basename(u))[0] not in already_downloaded_basenames]
  already_have_count = len(image_urls) - len(urls_to_download)
  if already_have_count > 0:
    print(f"⏭️ Skipping {already_have_count} image(s) already present in {images_folder}")

  scrape_file = os.path.join(config_folder, f"scrape_{project_subfolder}.txt")
  with open(scrape_file, "w") as f:
    f.write("\n".join(urls_to_download))

  if not urls_to_download:
    print("✅ All images already downloaded, nothing new to fetch.")
    return

  print(f"🌐 Saved links to {scrape_file}\n\n🔁 Downloading images...\n")
  old_img_count = len([f for f in os.listdir(images_folder) if f.lower().endswith(supported_types)])

  os.chdir(images_folder)
  !aria2c --console-log-level=warn -c -j 2 -x 1 -k 1M -s 1 --max-tries=15 --retry-wait=8 --max-concurrent-downloads=2 --header="User-Agent: {user_agent}" --header="Referer: https://gelbooru.com/" -i {scrape_file}

  # Gelbooru's CDN sometimes returns fake 404s under bursty concurrent load
  # even for valid URLs. Diff the URL list against what actually landed on
  # disk and retry just the missing ones, a few times, backing off each pass
  # at an even gentler rate.
  max_retry_passes = 3
  for retry_pass in range(1, max_retry_passes + 1):
    downloaded_basenames = {os.path.splitext(f)[0] for f in os.listdir(images_folder) if f.lower().endswith(supported_types)}
    still_missing = [u for u in urls_to_download if os.path.splitext(os.path.basename(u))[0] not in downloaded_basenames]
    if not still_missing:
      break
    print(f"\n🔁 Retry pass {retry_pass}/{max_retry_passes}: re-attempting {len(still_missing)} missing image(s)...")
    retry_file = os.path.join(config_folder, f"scrape_{project_subfolder}_retry.txt")
    with open(retry_file, "w") as f:
      f.write("\n".join(still_missing))
    time.sleep(10 * retry_pass)
    !aria2c --console-log-level=warn -c -j 1 -x 1 -k 1M -s 1 --max-tries=15 --retry-wait=8 --max-concurrent-downloads=1 --header="User-Agent: {user_agent}" --header="Referer: https://gelbooru.com/" -i {retry_file}

  new_img_count = len([f for f in os.listdir(images_folder) if f.lower().endswith(supported_types)])
  print(f"\n✅ Downloaded {new_img_count - old_img_count} images.")
  print(f"\n number of images in image_urls: {len(urls_to_download)} ")

  missing_count = len(urls_to_download) - (new_img_count - old_img_count)
  if missing_count > 0:
    print(f"⚠️ {missing_count} image(s) failed to download after retries (out of {len(urls_to_download)} attempted). Their tag files will be cleaned up next.")

download_images()

def delete_gifs(directory):
  try:
      files = os.listdir(directory)

      # Iterate through files and delete GIFs
      for file in files:
          file_path = os.path.join(directory, file)
          if file.lower().endswith(".gif") and os.path.isfile(file_path):
              os.remove(file_path)
              print(f"Deleted .gif file: {file_path}")

      print("Deletion completed.")
  except Exception as e:
      print(f"Error deleting GIFs: {e}")

delete_gifs(images_folder)

def delete_orphaned_tag_files(directory):
  """Removes .txt tag files that have no matching image file (e.g. the tags
  were written but the image itself failed to download)."""
  files = os.listdir(directory)
  image_stems = {os.path.splitext(f)[0] for f in files if os.path.splitext(f)[1].lower() in image_extensions}
  removed = 0
  for file in files:
    stem, ext = os.path.splitext(file)
    if ext.lower() == ".txt" and stem not in image_stems:
      os.remove(os.path.join(directory, file))
      removed += 1
  if removed:
    print(f"🧹 Removed {removed} orphaned .txt tag file(s) with no matching image.")

delete_orphaned_tag_files(images_folder)


#######################################################
##### STEP - Tagging
#######################################################

def clean_tags(tags_list):
    cleaned_tags = [t.strip() for t in tags_list.split(",")]
    cleaned_tags = [t.replace("_", " ") if len(t) > 3 else t for t in cleaned_tags]
    # Check if the list contains only an empty string
    if len(cleaned_tags) == 1 and cleaned_tags[0] == '':
        return []
    return cleaned_tags


#@markdown ---
#@markdown ### 4️⃣ Curate the Tags

#@markdown ❗ Important: you can choose to not enter anything in this section if you want to train your lora without a trigger

trigger = "" #@param {type:"string"}
#@markdown Abosrb tags that represent your Lora. This could be details like eye color or concepts like `glowing`
absorbed_these_tags_into_trigger = "" #@param {type:"string"}
#@markdown These tags will be removed from the captions - these are removed the same way the absorbed tags are
blacklist_tags = " absurdres, highres, bad id, bad pixiv id, " #@param {type:"string"}

%env PYTHONPATH=/env/python
os.chdir(root_dir)

blacklisted_tags = clean_tags(blacklist_tags)
absorbed_these_tags_into_trigger = clean_tags(absorbed_these_tags_into_trigger)
trigger_tags = clean_tags(trigger)

top_tags = Counter()
for txt in [f for f in os.listdir(images_folder) if f.lower().endswith(".txt")]:
  with open(os.path.join(images_folder, txt), 'r') as f:
    tags = [t.strip() for t in f.read().split(",")]
    tags = [t.replace("_", " ") if len(t) > 3 else t for t in tags]

    # Remove tags if they are in blacklisted_tags or absorbed_these_tags_into_trigger
    tags = [t for t in tags if t not in blacklisted_tags]
    tags = [t for t in tags if t not in absorbed_these_tags_into_trigger]

    # Add Triggers to beginning of tags

    for trigger_tag in trigger_tags:
      if trigger_tag in tags:
        tags.remove(trigger_tag)
      tags.insert(0, trigger_tag)

  top_tags.update(tags)
  with open(os.path.join(images_folder, txt), 'w') as f:
    f.write(", ".join(tags))


display(Markdown(f"### Here are the top 50 tags in your dataset"))
print("\n".join(f"{k}," for k, v in top_tags.most_common(50)))


print("-" * 50)
display(Markdown(f"### Here are Project stats"))
total_image_count = count_images_in_folder(images_folder)
print(f"Number of images in folder is {total_image_count}")

#######################################################
##### Zip with optional lossless WebP conversion
#######################################################

def zip_dataset_with_optional_webp(source_folder, zip_file_path, try_webp=True, webp_savings_threshold=0.20):
  """Zips source_folder as-is. If try_webp is True, also zips a lossless-WebP
  copy of all images (tag .txt files copied unchanged into both) and keeps
  the WebP version only if it's at least webp_savings_threshold smaller
  (e.g. 0.20 = 20%) than the original-format zip; otherwise keeps the
  original formats. If try_webp is False, skips the WebP pass entirely and
  always zips the original formats. Writes the winner to zip_file_path and
  returns which format was used ("webp" or "original"). source_folder itself
  is only ever read, never modified — original files on disk are left as-is
  regardless of which format ends up in the zip."""
  print(f"📦 Zipping dataset from {source_folder} ...")

  all_files = []
  for root, _, files in os.walk(source_folder):
    for file in files:
      all_files.append(os.path.join(root, file))
  image_count = sum(1 for f in all_files if os.path.splitext(f)[1].lower() in image_extensions)

  def build_zip(path, use_webp, show_progress=False):
    converted_count = 0
    file_iter = tqdm(all_files, desc="Converting to WebP", unit="file") if show_progress else all_files
    with zipfile.ZipFile(path, 'w', zipfile.ZIP_DEFLATED) as zipf:
      for file_path in file_iter:
        arcname = os.path.relpath(file_path, source_folder)
        ext = os.path.splitext(file_path)[1].lower()
        if use_webp and ext in image_extensions and ext != ".webp":
          try:
            with Image.open(file_path) as img:
              webp_arcname = os.path.splitext(arcname)[0] + ".webp"
              with tempfile.NamedTemporaryFile(suffix=".webp", delete=False) as tmp:
                img.save(tmp.name, "WEBP", lossless=True)
                zipf.write(tmp.name, webp_arcname)
              os.remove(tmp.name)
              converted_count += 1
            continue
          except Exception as e:
            print(f"⚠️ Could not convert {os.path.basename(file_path)} to WebP ({e}), using original")
        zipf.write(file_path, arcname)
    return converted_count

  print("  → Zipping original formats...")
  build_zip(zip_file_path, use_webp=False)

  if not try_webp:
    original_size = os.path.getsize(zip_file_path)
    print(f"  → WebP conversion disabled. Final zip: {original_size/1e6:.1f} MB")
    return "original"

  webp_zip_path = zip_file_path + ".webp_candidate.zip"

  print(f"  → Converting {image_count} image(s) to lossless WebP...")
  converted_count = build_zip(webp_zip_path, use_webp=True, show_progress=True)
  print(f"  → Converted {converted_count} image(s) to lossless WebP")

  original_size = os.path.getsize(zip_file_path)
  webp_size = os.path.getsize(webp_zip_path)
  savings = 1 - (webp_size / original_size) if original_size > 0 else 0
  savings_bytes = original_size - webp_size

  print(f"  → Original-format zip: {original_size/1e6:.1f} MB | Lossless-WebP zip: {webp_size/1e6:.1f} MB")
  if savings_bytes >= 0:
    print(f"  → WebP would save {savings_bytes/1e6:.1f} MB ({savings*100:.0f}%)")
  else:
    print(f"  → WebP would cost an extra {-savings_bytes/1e6:.1f} MB ({-savings*100:.0f}% larger)")

  if savings >= webp_savings_threshold:
    os.replace(webp_zip_path, zip_file_path)
    print(f"✅ Using lossless WebP — saved {savings_bytes/1e6:.1f} MB ({savings*100:.0f}%) vs original formats. Final zip: {webp_size/1e6:.1f} MB")
    return "webp"
  else:
    os.remove(webp_zip_path)
    print(f"↩️ Keeping original formats (WebP savings below the {webp_savings_threshold*100:.0f}% threshold). Final zip: {original_size/1e6:.1f} MB")
    return "original"

# Name the zip after the Hugging Face repo (e.g. "claude/pikachu" -> "pikachu.zip")
# so it lines up with what ends up on the Hub. Falls back to project_name if
# huggingface_repo_id isn't set.
zip_name = huggingface_repo_id.split("/")[-1] if huggingface_repo_id else project_name

os.makedirs(zipped_datasets_folder, exist_ok=True)
zip_file_path = os.path.join(zipped_datasets_folder, f"{zip_name}.zip")
dataset_zip_format = zip_dataset_with_optional_webp(images_folder, zip_file_path, try_webp=convert_to_webp)
print(f"Dataset zipped to {zip_file_path}")





import matplotlib.pyplot as plt
from collections import Counter

#@markdown ---
#@markdown ### 5️⃣ Chart the Tags
num_tags_to_show = 20  #@param {type:"integer"}
tags_to_ignore_in_chart = "uncensored, breasts, blush, penis, hetero, cum, pussy, nude, open mouth, pubic hair, highres" #@param {type:"string"}

# Extract tags and counts
tags = list(top_tags.keys())
counts = list(top_tags.values())

# Exclude specified tags from the chart
tags_to_ignore = set(map(str.strip, tags_to_ignore_in_chart.split(',')))
tags_and_counts = list(zip(tags, counts))
filtered_tags_and_counts = [(tag, count) for tag, count in tags_and_counts if tag not in tags_to_ignore]

# Sort tags and counts by counts in descending order
sorted_indices = sorted(range(len(filtered_tags_and_counts)), key=lambda k: filtered_tags_and_counts[k][1], reverse=True)

tags_to_show = [filtered_tags_and_counts[i][0] for i in sorted_indices[:num_tags_to_show]]
counts_to_show = [filtered_tags_and_counts[i][1] for i in sorted_indices[:num_tags_to_show]]

# Calculate percentages for each tag
percentages = [(count / total_image_count) * 100 for count in counts_to_show]

# Plot the bar chart for the top tags
plt.figure(figsize=(10, 6))
bars = plt.bar(tags_to_show, counts_to_show, color='blue')
plt.xlabel(f'Top {num_tags_to_show} Tag Occurrences (excluding specified tags)')
plt.ylabel('Number of Occurrences')
plt.title(f'{project_name}')
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better visibility

# Display percentages at the top of each bar
for bar, percentage in zip(bars, percentages):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 0.05 * total_image_count, f'{percentage:.0f}%', ha='center')

#plt.tight_layout()

chart_filename = os.path.join(main_dir, f"{zip_name}.png")
plt.savefig(chart_filename, bbox_inches='tight')

# Show the chart
plt.show()


#######################################################
##### STEP - Upload to Hugging Face Hub
#######################################################

if not upload_to_huggingface:
  print("skipping Hugging Face upload")
else:
  # Hardcoded README template, populated at runtime and shipped as-is to the HF
  # dataset repo. Edit this string to change what every uploaded dataset's
  # README looks like; users can hand-tweak the result afterwards on HF.
  README_TEMPLATE = """# {project_name} Dataset Summary

- **Gelbooru search tags:** `{gelbooru_search_tags}`
- **Number of images:** {total_image_count}
- **Created:** {created_date}

## Top Tags

```
{top_tags_list}
```

## Tag Distribution

![Tag chart]({chart_image_name})

## Notes
- The dataset is provided as a ZIP file
- Intended for AI training and research purposes
"""

  if not huggingface_repo_id:
    raise ValueError("Set huggingface_repo_id under section 2️⃣ before running this cell.")

  from google.colab import userdata
  try:
    hf_token = userdata.get('HF_TOKEN')
  except userdata.SecretNotFoundError as e:
    raise RuntimeError(
      "Missing Colab secret. Add HF_TOKEN via the key icon in the left sidebar."
    ) from e
  except userdata.NotebookAccessError as e:
    raise RuntimeError(
      "HF_TOKEN secret exists but notebook access is disabled. "
      "Enable notebook access for it in the left sidebar."
    ) from e

  chart_image_name = f"{zip_name}.png"
  top_tags_list = ", ".join(k for k, v in top_tags.most_common(20))

  # Populate the README template
  readme_content = README_TEMPLATE.format(
    project_name=project_name,
    gelbooru_search_tags=gelbooruSearchQuery.strip(),
    total_image_count=total_image_count,
    created_date=datetime.now().strftime('%Y-%m-%d'),
    top_tags_list=top_tags_list,
    chart_image_name=chart_image_name,
  )
  hf_readme_path = os.path.join(zipped_datasets_folder, "README.md")
  with open(hf_readme_path, "w") as f:
    f.write(readme_content)
  print(f"📝 Wrote README to {hf_readme_path}")

  !pip install -q -U huggingface_hub
  from huggingface_hub import HfApi

  print(f"☁️ Uploading {zip_name}.zip, README.md, and tag chart to Hugging Face dataset repo '{huggingface_repo_id}'...")
  api = HfApi(token=hf_token)
  api.create_repo(repo_id=huggingface_repo_id, repo_type="dataset", private=huggingface_repo_private, exist_ok=True)
  api.upload_file(path_or_fileobj=zip_file_path, path_in_repo=f"{zip_name}.zip", repo_id=huggingface_repo_id, repo_type="dataset", token=hf_token)
  api.upload_file(path_or_fileobj=hf_readme_path, path_in_repo="README.md", repo_id=huggingface_repo_id, repo_type="dataset", token=hf_token)
  if os.path.exists(chart_filename):
    api.upload_file(path_or_fileobj=chart_filename, path_in_repo=chart_image_name, repo_id=huggingface_repo_id, repo_type="dataset", token=hf_token)
  else:
    print("⚠️ Tag chart file not found — README image link will be broken until one is uploaded.")
  print(f"✅ Uploaded dataset to https://huggingface.co/datasets/{huggingface_repo_id}")